# Global GC* 

In [ ]:
import os
from scipy import signal, stats
import numpy as np

# os.environ["MODIN_ENGINE"] = "ray"  # Modin will use Ray
# import modin.pandas as pd
import pandas as pd

import bisect
import importlib
import warnings
from pathlib import Path
import gc

from tqdm.notebook import trange, tqdm

import ete3
np.random.seed(7)

# import psutil, os
# def mem():
#     return psutil.Process(os.getpid()).memory_info().rss / 1024**3

Plotting setup:

In [ ]:
%config InlineBackend.figure_formats = ['retina', 'png']
#%config InlineBackend.figure_formats = ['svg']
%matplotlib inline

# # Make inline plots vector graphics instead of raster graphics
# from matplotlib_inline.backend_inline import set_matplotlib_formats
# #set_matplotlib_formats('pdf', 'svg')
# set_matplotlib_formats('retina', 'png')

import matplotlib
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import Rectangle
from matplotlib.lines import Line2D
from matplotlib import gridspec

#matplotlib.rcParams['figure.figsize'] = (20.0, 10.0)

# import mpld3

import seaborn as sns
sns.set() # sets seaborn default "prettyness:
sns.set_style("white")
sns.set_context("notebook")

# lowess for plotting
from statsmodels.nonparametric.smoothers_lowess import lowess

# My own paired palette replacing the last brown pair with violets
sns.color_palette('Paired').as_hex()
Paired = sns.color_palette(['#a6cee3', '#1f78b4', '#b2df8a', '#33a02c', '#fb9a99', '#e31a1c',
                            '#fdbf6f', '#ff7f00', '#cab2d6','#6a3d9a', '#e585cf', '#ad009d'])
#sns.palplot(Paired)

from shared_vars_and_func import *

Monospace font for numbers in tables:

In [ ]:
%%html
<style> table { font-variant-numeric: tabular-nums; } </style>

Paths to data and results:

In [ ]:
data_path = Path('/project/Birds/faststorage/data')
results_path = Path('/project/Birds/faststorage/people/kmt/bird-hotspots/results')

Define column sets:

In [ ]:
base_counts = ['nA', 'nG', 'nT', 'nC']

substitution_counts =  ['nA2C', 'nA2G', 'nA2T', 'nC2A', 'nC2G', 'nC2T',
                        'nG2A', 'nG2C', 'nG2T', 'nT2A', 'nT2C', 'nT2G']
cpg_substitution_counts =  ['nA2c', 'nA2g', 'nc2A', 'nc2g', 'nc2T',
                            'ng2A', 'ng2c', 'ng2T', 'nT2c', 'nT2g']
# in paired order
substitutions = ['rA2C', 'rT2G',
                 'rC2T', 'rG2A',
                 'rA2G', 'rT2C',
                 'rA2T', 'rT2A', 
                 'rC2A', 'rG2T',
                 'rC2G', 'rG2C']
# cpg_substitutions = ['rA2c', 'rT2g',
#                      'rc2T', 'rg2A',
#                      'rA2g', 'rT2c',
#                      'rc2A', 'rg2T',
#                      'rc2g', 'rg2c']


transitions = ['rA2G', 'rG2A', 'rT2C', 'rC2T']
# cpg_transitions = ['rA2g', 'rg2A', 'rT2c', 'rc2T']

transversions = [x for x in substitutions if x not in transitions]
# cpg_transversions = [x for x in cpg_substitutions if x not in cpg_transitions]

paired_patterns = [('rT2G', 'rA2C'),
                   ('rA2G', 'rT2C'),
                   ('rA2T', 'rT2A'), 
                   ('rG2T', 'rC2A'),
                   ('rC2G', 'rG2C'), 
                   ('rC2T', 'rG2A')]
# cpg_paired_patterns = [('rT2g', 'rA2c'),
#                        ('rA2g', 'rT2c'),
#                        ('rg2T', 'rc2A'),
#                        ('rc2g', 'rg2c'), 
#                        ('rc2T', 'rg2A')]

chromosomes = ['1', '1A', '2', '3', '4', '4A', '5', '6', '7', '9', '10', '11', '12', '13', '14', '15']

# Load species details

In [ ]:
species_details = pd.read_csv('/home/kmt/Birds/faststorage/data/species_details.txt', sep='\t')
species_details.head()

# GC* by chromosome of each species

Mean GC* for each chromosome of each species:

In [ ]:
chromosome_GCstar = pd.read_hdf('../results/chromosome_GCstar.h5')
chromosome_GCstar.head()

Size order of TAEGU chromosomes:

In [ ]:
taegu_chromosome_sizes = pd.read_hdf('../results/taegu_chromosome_sizes.h5').sort_values('chrom_size')
taegu_chromosome_sizes.head()

Plot chromosome GC* across species (ordered by TAEGU chromosome size):

In [ ]:
taegu_chromosome_size_order = taegu_chromosome_sizes.sort_values('chrom_size').chrom

In [ ]:
with sns.axes_style('whitegrid'):
    g = sns.FacetGrid(data=chromosome_GCstar, col='species',
                      col_order=chromosome_GCstar.species.unique().sort(),
                      col_wrap=8, sharex=True, sharey=True)
    g.map(sns.pointplot, "chrom", 'GCstar', errorbar=None, order=taegu_chromosome_size_order)
    g.set_titles(col_template="{col_name}") ;

In [ ]:
taegu_chromosome_gcstar_order = chromosome_GCstar.loc[chromosome_GCstar.species_code == 'TAEGU'].sort_values('GCstar').chrom

In [ ]:
with sns.axes_style('whitegrid'):
    g = sns.FacetGrid(data=chromosome_GCstar, col='species', 
                      col_order=chromosome_GCstar.species.unique().sort(),
                      col_wrap=8, sharex=True, sharey=True)
    g.map(sns.pointplot, "chrom", 'GCstar', errorbar=None, order=taegu_chromosome_gcstar_order)
    g.set_titles(col_template="{col_name}") ;

### TODO: is the difference to TAEGU in rank ordering related to the amount of hotspot sharing?

log2 ratio of GC* of largest half vs smallest half of (TAEGU ordered) chromosomes:

In [ ]:
def small_vs_large_log2ratio(df):
    large = ['1', '1A', '2', '3', '4', '4A', '5', '6']
    small = ['7', '9', '10', '11', '12', '13', '14', '15']
    return np.log10(df.loc[df.chrom.isin(small), 'GCstar'].mean() / df.loc[df.chrom.isin(large), 'GCstar'].mean())

chromosome_GCstar.groupby('species').apply(small_vs_large_log2ratio)

# 200kb window GC*

In [ ]:
gcstar_200kb_windows = pd.read_hdf('../results/gcstar_200kb_windows.h5')
gcstar_200kb_windows.head()

In [ ]:
#gcstar_200kb_windows.groupby(['species', 'species_code']).window_gcstar.mean()

In [ ]:
plt.hist(gcstar_200kb_windows.window_gcstar, bins=30) ;

In [ ]:
gcstar_200kb_windows[['species', 'species_code', 'window_gcstar']].head(4100)

In [ ]:
g = sns.kdeplot(data=gcstar_200kb_windows.reset_index(), x="window_gcstar", 
            hue="species", legend=False, linewidth=0.5) ;

In [ ]:
g = sns.kdeplot(data=gcstar_200kb_windows.loc[gcstar_200kb_windows.species_code.isin(['TAEGU', 'HALLE', 'PICPU'])].reset_index(), x="window_gcstar", 
            hue="species", legend=False, linewidth=2) ;